# **Итоговый пайплайн и посылка решения**
#### Для Kaggle соревнования [**Dota 2 - HSE ML-1 Course Competition 2026**](https://www.kaggle.com/competitions/dota-2-hse-ml-1-course-competition-2026/overview) в рамках курса "Машинное обучение-1" на ПМИ, ВШЭ ФКН.

Соберем воедино все мысли и идеи из ноутбука `base-eda-and-feature-engineering.ipynb` в один красивый и лаконичный пайплайн.

Блок с импортами + объявление $\text{Gini}$:

In [29]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer,
    TargetEncoder
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_score
from sklearn.metrics import make_scorer, roc_auc_score, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

import optuna

from typing import Iterable, Literal

In [30]:
def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0

### **0. Загрузка датасета**

Вдруг он потерялся или с ним что-то случилось. Или просто первый ноутбук не запускался. Лишним не будет.

In [31]:
# Для этого надо установить kaggle токен. Как это сделать, вот: https://github.com/Kaggle/kaggle-cli
# Ну или самому закинуть zip-ник и забить на этот блок

%pip install kaggle -q
!kaggle competitions download -c dota-2-hse-ml-1-course-competition-2026

Note: you may need to restart the kernel to use updated packages.
dota-2-hse-ml-1-course-competition-2026.zip: Skipping, found more recently modified local copy (use --force to force download)


In [32]:
import zipfile
import os

data_folder_name = 'data_folder'

with zipfile.ZipFile('dota-2-hse-ml-1-course-competition-2026.zip', 'r') as zip_ref:
    zip_ref.extractall(data_folder_name)

### **1. Сам пайпалайн**

Соберем воедино все кастомные трасформеры, которые писались до этого/написаны сейчас.

Преобразование дат:

In [33]:
def get_day_of_week(df):
    dates = pd.to_datetime(df.iloc[:, 0])
    return dates.dt.dayofweek.values.reshape(-1, 1)

Лоадоры для статистики игроков и матчевой статистики:

In [34]:
class PlayersLoader(BaseEstimator, TransformerMixin):
    def __init__(self, players_df: pd.DataFrame):
        self.players_df = players_df
        self.clean_players_df_ = None

    def _prepare_players_df(self):
        df = self.players_df.copy()

        df = df[(df["account_id"] != -1) & (df["hero_id"] != 0)]

        hero_counts = df.groupby(["match_id", "hero_id"]).size().reset_index(name='count')
        bad_matches_dups = hero_counts[hero_counts['count'] > 1]['match_id'].unique()
        df = df[~df["match_id"].isin(bad_matches_dups)]

        df["team_sign"] = np.where(df["player_slot"].isin(range(0, 5)), 1, -1)
        team_counts = df.groupby(["match_id", "team_sign"]).size().reset_index(name='p_count')
        bad_matches_incomplete = team_counts[team_counts['p_count'] != 5]['match_id'].unique()
        df = df[~df["match_id"].isin(bad_matches_incomplete)]

        return df

    def fit(self, X, y=None):
        self.clean_players_df_ = self._prepare_players_df()
        self.is_fitted_ = True
        return self

    def transform(self, X):
        match_ids = X['match_id'].tolist()

        extracted = self.clean_players_df_[self.clean_players_df_['match_id'].isin(match_ids)].copy()

        extracted['match_id'] = pd.Categorical(
            extracted['match_id'],
            categories=match_ids,
            ordered=True
        )

        index_map = dict(zip(X['match_id'], X.index))
        extracted['_orig_index'] = extracted['match_id'].map(index_map)

        return extracted

    def get_feature_names_out(self, input_features=None):
        if self.clean_players_df_ is not None:
            return self.clean_players_df_.columns.values
        return np.array([])

class AdvLoader(BaseEstimator, TransformerMixin):
    def __init__(self, dota_adv_df, columns):
        self.dota_adv_df = dota_adv_df
        self.columns = columns
        self.dota_adv_df_indexed = None

    def _parse_str_to_list(self, s):
            s = s.strip("[]")
            return np.array([int(x) for x in s.split()])

    def fit(self, X, y=None):
        self.dota_adv_df_indexed = self.dota_adv_df.set_index('match_id')
        self.is_fitted_ = True
        return self

    def transform(self, X):
        match_ids = X.iloc[:, 0]
        extracted = self.dota_adv_df_indexed.reindex(match_ids).reset_index(drop=True)

        for col in self.columns:
            extracted[col] = extracted[col].apply(self._parse_str_to_list)
        return extracted[self.columns]

    def get_feature_names_out(self, input_features=None):
        return np.array([])


Эктрактор, который кодирует вектор героев:

In [35]:
class HeroesVectorsExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, all_heroes_ids: Iterable[int]):
        self.all_heroes_ids = all_heroes_ids
        self.hero_columns = [f'hero_{i}' for i in all_heroes_ids]

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        vectors = X.pivot_table(
            index='match_id',
            columns='hero_id',
            values='team_sign',
            aggfunc='sum',
            fill_value=0,
            dropna=False,
            observed=False
        )

        vectors.columns = [f'hero_{int(col)}' for col in vectors.columns]

        missing_cols = list(set(self.hero_columns) - set(vectors.columns))
        if missing_cols:
            zeros_df = pd.DataFrame(0, index=vectors.index, columns=missing_cols)
            vectors = pd.concat([vectors, zeros_df], axis=1)

        return vectors[self.hero_columns].astype(int).reset_index(drop=True)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.hero_columns)

Эктракторы для фичей по статистике внутреигровых показателей:

In [36]:
class TrendFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, columns: Iterable[str], method: Literal["delta", "OLS"] = "OLS"):
        self.columns = columns
        self.method = method
        self.feature_names = [f"{col}_{m}" for col in columns for m in ["slope", "intercept", "r2"]]

    def _get_trend_stats(self, adv_list):
        if len(adv_list) == 0:
            return pd.Series({'slope': 0, 'intercept': 0, 'r2': 0})
        slope = 0
        intercept = 0
        r2 = 0

        if self.method == "delta":
            slope = (adv_list[-1] - adv_list[0]) / len(adv_list)
            intercept = adv_list[0]
        elif self.method == "OLS":
            y = adv_list
            x = np.arange(len(adv_list))
            slope, intercept = np.polyfit(x, y, 1)
            y_pred = slope * x + intercept
            r2 = r2_score(y, y_pred)

        return pd.Series({'slope': slope, 'intercept': intercept, 'r2': r2})

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        all_stats = []
        for col in self.columns:
            stats = X[col].apply(self._get_trend_stats)
            stats.columns = [f"{col}_slope", f"{col}_intercept", f"{col}_r2"]
            all_stats.append(stats)
        return pd.concat(all_stats, axis=1).fillna(0)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names)


class BinStatisticExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, columns: Iterable[str], n_bins: int = 3, add_last_value: bool = True):
        self.columns = columns
        self.n_bins = n_bins
        self.add_last_value = add_last_value

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X, y=None):
        new_features = []

        for col in self.columns:
            stats = X[col].apply(self._get_binned_stats)
            stats.columns = [f"{col}_{c}" for c in stats.columns]
            new_features.append(stats)

            if self.add_last_value:
                last_val = X[col].apply(lambda x: x[-1] if len(x) > 0 else 0)
                last_val.name = f"{col}_last_adv_value"
                new_features.append(last_val)

        return pd.concat(new_features, axis=1).fillna(0)

    def _get_binned_stats(self, adv_list):
        res = {}
        if len(adv_list) == 0:
            for i in range(self.n_bins):
                res.update({
                    f"bin_{i}_mean": 0,
                    f"bin_{i}_std": 0,
                    f"bin_{i}_max_dire_lead": 0,
                    f"bin_{i}_max_radiant_lead": 0,
                    f"bin_{i}_lead_switches": 0,
                })
            return pd.Series(res)

        bins = np.array_split(adv_list, self.n_bins)
        for i, adv_bin in enumerate(bins):
            bin_stats = self._get_bin_advanced_stat(adv_bin)
            for stat_name, stat_val in bin_stats.items():
                res[f"bin_{i}_{stat_name}"] = stat_val

        return pd.Series(res)

    def _get_bin_advanced_stat(self, adv_bin):
        if len(adv_bin) == 0:
            return {
                "mean": 0,
                "std": 0,
                "max_dire_lead": 0,
                "max_radiant_lead": 0,
                "lead_switches": 0,
            }

        return {
            "mean": np.mean(adv_bin),
            "std": np.std(adv_bin),
            "max_dire_lead": -np.min(adv_bin),
            "max_radiant_lead": np.max(adv_bin),
            "lead_switches": self._count_zero_crossings(adv_bin),
        }

    @staticmethod
    def _count_zero_crossings(arr):
        signs = np.sign(arr)
        signs = signs[signs != 0]
        crossings = np.where(np.diff(signs))[0]
        return len(crossings)

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for col in self.columns:
            for i in range(self.n_bins):
                feature_names.extend([
                    f"{col}_bin_{i}_mean",
                    f"{col}_bin_{i}_std",
                    f"{col}_bin_{i}_max_dire_lead",
                    f"{col}_bin_{i}_max_radiant_lead",
                    f"{col}_bin_{i}_lead_switches",
                ])
            if self.add_last_value:
                feature_names.append(f"{col}_last_adv_value")
        return np.array(feature_names)

Эктрактор фичей из статистики игроков в матче:

In [37]:
class PlayersStatisticExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, delta_columns: Iterable[str], min_max_columns: Iterable[str], add_kda: bool = True):
        self.delta_columns = delta_columns
        self.min_max_columns = min_max_columns
        self.add_kda = add_kda
        self.delta_cols = None
        self.min_max_cols = None
        self.statistic_columns = None

    def _fill_col_names(self):
        self.delta_cols = list(self.delta_columns)
        self.min_max_cols = list(self.min_max_columns)

        self.statistic_columns = [f"total_delta_{col}" for col in self.delta_cols]

        for col in self.min_max_cols:
            self.statistic_columns.extend([
                f"max_{col}_radiant", f"min_{col}_radiant",
                f"max_{col}_dire", f"min_{col}_dire"
            ])

        if self.add_kda:
            self.statistic_columns.extend(["kda_radiant", "kda_dire", "delta_kda"])

    def fit(self, X, y=None):
        self._fill_col_names()
        self.is_fitted_ = True
        return self

    def transform(self, X):
        radiant_df = X[X['team_sign'] == 1]
        dire_df = X[X['team_sign'] == -1]

        rad_sum = radiant_df.groupby('match_id', observed=False)[self.delta_cols].sum()
        dire_sum = dire_df.groupby('match_id', observed=False)[self.delta_cols].sum()

        delta_df = rad_sum - dire_sum
        delta_df.columns = [f"total_delta_{col}" for col in self.delta_cols]

        if self.min_max_cols:
            rad_max = radiant_df.groupby('match_id', observed=False)[self.min_max_cols].max()
            rad_min = radiant_df.groupby('match_id', observed=False)[self.min_max_cols].min()
            dire_max = dire_df.groupby('match_id', observed=False)[self.min_max_cols].max()
            dire_min = dire_df.groupby('match_id', observed=False)[self.min_max_cols].min()

            rad_max.columns = [f"max_{col}_radiant" for col in self.min_max_cols]
            rad_min.columns = [f"min_{col}_radiant" for col in self.min_max_cols]
            dire_max.columns = [f"max_{col}_dire" for col in self.min_max_cols]
            dire_min.columns = [f"min_{col}_dire" for col in self.min_max_cols]

            min_max_df = pd.concat([rad_max, rad_min, dire_max, dire_min], axis=1)
        else:
            min_max_df = pd.DataFrame(index=delta_df.index)

        if self.add_kda:
            kda_cols = ['kills', 'assists', 'deaths']
            if all(c in X.columns for c in kda_cols):
                rad_kda = radiant_df.groupby('match_id', observed=False)[kda_cols].sum()
                dire_kda = dire_df.groupby('match_id', observed=False)[kda_cols].sum()

                kda_rad = (rad_kda['kills'] + rad_kda['assists']) / (rad_kda['deaths'] + 1)
                kda_dire = (dire_kda['kills'] + dire_kda['assists']) / (dire_kda['deaths'] + 1)

                kda_df = pd.DataFrame({
                    'kda_radiant': kda_rad,
                    'kda_dire': kda_dire,
                    'delta_kda': kda_rad - kda_dire
                }, index=delta_df.index)
            else:
                kda_df = pd.DataFrame(0, index=delta_df.index, columns=["kda_radiant", "kda_dire", "delta_kda"])
        else:
            kda_df = pd.DataFrame(index=delta_df.index)

        result = pd.concat([delta_df, min_max_df, kda_df], axis=1)
        result = result[self.statistic_columns].fillna(0)

        orig_indices = X.groupby('match_id', observed=False)['_orig_index'].first().values
        result.index = orig_indices

        return result

    def get_feature_names_out(self, input_features=None):
        if self.statistic_columns is None:
            return np.array([])
        return np.array(self.statistic_columns)

В процессе дебага низкого качества на тесте было выяснено, что `players_df` не содержит нужной нам информации про тестовые матчи, что очень печально (это показано в конце ноутбука про EDA). Эти сведенья были бы очень полезны для оценки вероятности победы в матче. 

Код выше оставлен так как уже был написан и отдебажен на корректность функционала. Такое жалко удалять.

Эктрактор фичей по типу атаки героев:

In [38]:
class HeroAttackTypeExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, heroes_df: pd.DataFrame):
        self.heroes_df = heroes_df
        self.attack_type_count_columns = None

    def _fill_col_names(self):
        unique_attack_types = self.heroes_df["attack_type"].dropna().unique()

        self.attack_type_count_columns = []
        for attack_type in unique_attack_types:
            self.attack_type_count_columns.extend([
                f"{attack_type}_count_radiant",
                f"{attack_type}_count_dire"
            ])

    def fit(self, X, y=None):
        self._fill_col_names()
        self.is_fitted_ = True
        return self

    def transform(self, X):
        X_mapped = X.copy()
        attack_type_map = self.heroes_df.set_index('id')['attack_type']
        X_mapped['attack_type'] = X_mapped['hero_id'].map(attack_type_map)

        radiant_df = X_mapped[X_mapped['team_sign'] == 1]
        dire_df = X_mapped[X_mapped['team_sign'] == -1]

        rad_counts = radiant_df.groupby(['match_id', 'attack_type'], observed=False).size().unstack(fill_value=0)
        dire_counts = dire_df.groupby(['match_id', 'attack_type'], observed=False).size().unstack(fill_value=0)

        rad_counts.columns = [f"{col}_count_radiant" for col in rad_counts.columns]
        dire_counts.columns = [f"{col}_count_dire" for col in dire_counts.columns]

        result = pd.concat([rad_counts, dire_counts], axis=1)
        result = result.reindex(columns=self.attack_type_count_columns, fill_value=0)

        orig_indices = X.groupby('match_id', observed=False)['_orig_index'].first().values
        result.index = orig_indices

        return result

    def get_feature_names_out(self, input_features=None):
        if self.attack_type_count_columns is None:
            np.array([])
        return np.array(self.attack_type_count_columns)

А теперь сам пайплайн.

In [39]:
def build_dota_pipeline(
    players_df: pd.DataFrame,
    heroes_df: pd.DataFrame,
    dota_adv_df: pd.DataFrame,
    all_heroes_ids: Iterable[int] | None = None,
    region_te_smooth: float | str = "auto",
    trend_method: Literal["delta", "OLS"] = "OLS",
    n_adv_bins: int = 3,
    max_iter: int = 2000,
    logistic_regression_solver: str = 'lbfgs',
    penalty: str = 'l2',
    C: float = 1.0
):
    region_features = ["region"]
    region_transformer = Pipeline([
        ('region_te', TargetEncoder(target_type='binary', smooth=region_te_smooth)),
        ('scaler', StandardScaler())
    ])

    date_features = ["date"]
    date_transformer = Pipeline([
        ('extract_day', FunctionTransformer(get_day_of_week, feature_names_out=lambda tf, names_in: ["day_of_week"])),
        ('day_of_week_ohe', OneHotEncoder(categories=[[0, 1, 2, 3, 4, 5, 6]], handle_unknown='ignore', drop='first'))
    ])

    mmr_features = ["avg_mmr"]
    mmr_transformer = Pipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=42, add_indicator=True)),
        ('log_transform', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scaler', StandardScaler())
    ])

    join_keys = ["match_id"]
    if all_heroes_ids is None:
        all_heroes_ids = heroes_df["id"].unique()

    heroes_extractor_pipeline = Pipeline([
        ('loader', PlayersLoader(players_df=players_df)),
        ('extractors', FeatureUnion([
            ('hero_vectorizer', HeroesVectorsExtractor(all_heroes_ids=all_heroes_ids)),
            ('team_attack_type_ohe', Pipeline([
                ('team_attack_type', HeroAttackTypeExtractor(heroes_df=heroes_df)),
                ('attack_type_ohe', OneHotEncoder(categories=[[0, 1, 2, 3, 4, 5]] * 4, handle_unknown='ignore', drop='first'))
            ]))
        ])),
    ])

    adv_cols = ['radiant_gold_adv', 'radiant_exp_adv']
    adv_extractor_pipeline = Pipeline([
        ('loader', AdvLoader(dota_adv_df=dota_adv_df, columns=adv_cols)),
        ('extractors', FeatureUnion([
            ('trends', TrendFeatureExtractor(columns=adv_cols, method=trend_method)),
            ('bins_stats', BinStatisticExtractor(columns=adv_cols, n_bins=n_adv_bins, add_last_value=True))
        ])),
        ('scaler', StandardScaler())
    ])

    column_tranformer = ColumnTransformer(
        transformers=[
            ('region_transformer', region_transformer, region_features),
            ('date_transformer', date_transformer, date_features),
            ('mmr_transformer', mmr_transformer, mmr_features),
            ('heroes_extractor_pipeline', heroes_extractor_pipeline, join_keys),
            ('adv_extractor_pipeline', adv_extractor_pipeline, join_keys),
        ],
        remainder='drop'
    )
    full_pipeline = Pipeline([
        ('column_transformer', column_tranformer),
        ('classifier', LogisticRegression(max_iter=max_iter, solver=logistic_regression_solver, penalty=penalty, C=C))
    ])

    return full_pipeline

### **2. Тестирование пайплайна**

Считаем все необходимые данные. 

In [40]:
data_folder_name = "data_folder"
target = "radiant_win"

df_train = pd.read_csv(f"{data_folder_name}/matches_df_train.csv")
df_test = pd.read_csv(f"{data_folder_name}/matches_df_test.csv")
players_df = pd.read_csv(f"{data_folder_name}/player_df.csv")
heroes_df = pd.read_csv(f"{data_folder_name}/Constants.Heroes.csv")
dota_adv_df = pd.read_csv(f"{data_folder_name}/dota_adv.csv")

X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test = df_test

all_heroes_ids = heroes_df["id"].unique()

In [41]:
X_train["date"] = pd.to_datetime(X_train["date"])
X_train = X_train.sort_values("date")
y_train = y_train.loc[X_train.index]

А теперь просто тестовый запуск пайплайна, чтобы убедиться, что он работает и выдает предсказания нужного формата.

In [42]:
my_pipeline = build_dota_pipeline(
    players_df=players_df,
    heroes_df=heroes_df,
    dota_adv_df=dota_adv_df,
    all_heroes_ids=all_heroes_ids
)

my_pipeline.fit(X_train[:1000], y_train[:1000])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('column_transformer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('region_transformer', ...), ('date_transformer', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If

In [43]:
preprocessor = my_pipeline.named_steps['column_transformer']
feature_names_out = preprocessor.get_feature_names_out()
print(f"Количество выходных фичей: {len(feature_names_out)}")

Количество выходных фичей: 193


In [44]:
model_predictions = my_pipeline.predict_proba(X_test[:1000])[:, 1]
model_predictions

array([1.79049653e-01, 5.70032049e-01, 6.40602275e-01, 4.04730014e-01,
       2.79865190e-01, 7.68718829e-01, 3.29403168e-01, 6.52092352e-01,
       4.83205273e-01, 6.73854719e-01, 5.11255538e-02, 5.28570477e-01,
       4.69795326e-01, 5.87276494e-01, 8.13933853e-01, 8.72848713e-01,
       7.36795759e-01, 2.25455980e-01, 6.89784732e-01, 8.48219410e-01,
       6.90056009e-01, 4.10590444e-01, 1.87178920e-01, 8.68581845e-01,
       4.43341604e-01, 9.43947723e-01, 5.56094798e-01, 6.42989771e-01,
       7.74574265e-01, 6.53676756e-01, 8.15840882e-01, 9.21103487e-01,
       7.74810322e-01, 3.26740132e-01, 5.63358141e-01, 1.15756900e-01,
       5.11469883e-01, 6.65067267e-01, 1.14631843e-01, 2.54936932e-01,
       2.07357675e-01, 6.26529897e-01, 9.27763539e-01, 6.50205411e-01,
       1.60475569e-01, 6.14452943e-01, 2.65448491e-01, 6.75604336e-02,
       8.29404578e-01, 7.91790272e-01, 1.91243167e-01, 6.45061256e-01,
       3.58870767e-01, 8.81508299e-01, 8.73646311e-01, 6.95245210e-01,
      

Ура работает! Теперь можно обучать на всем датасете и делать предсказания для сабмита.

### **3. Подбор гиперпараметров**

In [45]:
import warnings

n = 100000
random_indices = np.random.choice(len(X_train), size=n, replace=False)
sorted_indices = np.sort(random_indices)

X_sampled = X_train.iloc[sorted_indices]
y_sampled = y_train.iloc[sorted_indices]

split_idx = int(len(X_sampled) * 0.8)
X_sub_train, X_sub_val = X_sampled.iloc[:split_idx], X_sampled.iloc[split_idx:]
y_sub_train, y_sub_val = y_sampled.iloc[:split_idx], y_sampled.iloc[split_idx:]

column_transformer = build_dota_pipeline(
    players_df=players_df,
    heroes_df=heroes_df,
    dota_adv_df=dota_adv_df,
    all_heroes_ids=all_heroes_ids,
    region_te_smooth = "auto",
    trend_method = "OLS",
    n_adv_bins = 3
)["column_transformer"]

X_sub_train_transformed = column_transformer.fit_transform(X_sub_train, y_sub_train)
X_sub_val_transformed = column_transformer.transform(X_sub_val)

In [46]:
def objective(trial):
    warnings.filterwarnings("ignore")

    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    if penalty == "l1":
        solver = trial.suggest_categorical("solver_l1", ["liblinear", "saga"])
    else:
        solver = trial.suggest_categorical("solver_l2", ["lbfgs", "saga"])
    C = trial.suggest_float("C", 1e-4, 10.0, log=True)

    model = LogisticRegression(max_iter=3000, solver=solver, penalty=penalty, C=C)

    model.fit(X_sub_train_transformed, y_sub_train)
    probs = model.predict_proba(X_sub_val_transformed)[:, 1]

    return gini(y_sub_val, probs)

study = optuna.create_study(
    study_name="dota",
    direction="maximize"
)

study.optimize(objective, show_progress_bar=True, n_trials=50, n_jobs=-1)

[I 2026-06-24 18:23:30,538] A new study created in memory with name: dota


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-24 18:23:30,955] Trial 11 finished with value: 0.3834644285639883 and parameters: {'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 0.0005291143975140576}. Best is trial 11 with value: 0.3834644285639883.
[I 2026-06-24 18:23:30,991] Trial 3 finished with value: 0.39191129776512 and parameters: {'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 0.0008748197496463862}. Best is trial 3 with value: 0.39191129776512.
[I 2026-06-24 18:23:31,221] Trial 6 finished with value: 0.4027239420756603 and parameters: {'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 0.004658405357921346}. Best is trial 6 with value: 0.4027239420756603.
[I 2026-06-24 18:23:31,884] Trial 12 finished with value: 0.40419474513372977 and parameters: {'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 0.009373932524025711}. Best is trial 12 with value: 0.40419474513372977.
[I 2026-06-24 18:23:32,532] Trial 4 finished with value: 0.4072930394145198 and parameters: {'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 5.094828626109906}. Best is

In [47]:
best_params = study.best_params
best_params

{'penalty': 'l2', 'solver_l2': 'lbfgs', 'C': 0.8694194454067903}

### **4. Формирование и посылка submission'а**

In [49]:
model = build_dota_pipeline(
    players_df=players_df,
    heroes_df=heroes_df,
    dota_adv_df=dota_adv_df,
    all_heroes_ids=all_heroes_ids,
    region_te_smooth = "auto",
    trend_method = "OLS",
    n_adv_bins = 3,
    penalty="l2",
    logistic_regression_solver="lbfgs",
    C=0.8694194454067903
)

In [50]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('column_transformer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('region_transformer', ...), ('date_transformer', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If

In [51]:
model_predictions = model.predict_proba(X_test)[:, 1]

In [52]:
submission = pd.DataFrame({
    'ID': X_test['match_id'],
    'Value': model_predictions
})

submission.to_csv("submission.csv", index=False)

pd.read_csv("submission.csv").head()

,ID,Value
0,8,0.428617
1,29,0.598995
2,34,0.605675
3,36,0.495282
4,61,0.584577


In [53]:
!kaggle competitions submit -c dota-2-hse-ml-1-course-competition-2026 -f submission.csv  -m "submission from pipeline without optuna tuning"

100%|██████████████████████████████████████| 1.48M/1.48M [00:01<00:00, 1.14MB/s]
Successfully submitted to Dota 2 - HSE ML-1 Course Competition 2026

In [ ]:
gini(y_train, model.predict_proba(X_train)[:, 1])